# Step 2 — First API Call

**Goal:** retrieve and print one real closing price for one ticker.

No loop, no files, no cloud — those are steps 3 and 5. This notebook answers a single
question: is the data reachable at all?

---

### Why a notebook here and a script afterwards

A notebook makes it cheap to run four lines, look at what came back, and adjust. That is
what the first encounter with an unfamiliar API response actually needs.

Notebooks cannot be scheduled, though. Step 10 hands this pipeline to a scheduler, and
schedulers run `.py` files. So the working pattern is:

> explore in a notebook -> move the working code into a script -> schedule the script

`fetch_one_ticker.py` sits beside this file and holds the version that graduates out of here.

**Rate limit:** the free tier allows 5 API calls per minute, and re-running a cell that hits
the API counts against it. An HTTP `429` means the limit was reached — wait sixty seconds.

---
## 1. Setup

Imports, plus a check that the notebook is running against the project's virtual environment.

In [1]:
import os
import sys
from datetime import datetime, timezone

import requests
from dotenv import load_dotenv

print("Imports OK")
print("Python:", sys.version.split()[0])
print("requests:", requests.__version__)

Imports OK
Python: 3.13.3
requests: 2.34.2


**What each import is for:**

| Import | Purpose |
|---|---|
| `os` | Reading environment variables, via `os.getenv()`. |
| `sys` | Reporting which Python is running. |
| `datetime`, `timezone` | Converting the API's epoch timestamps into readable dates. |
| `requests` | HTTP requests — the standard Python library for talking to web APIs. |
| `dotenv` | Loads a `.env` file into the environment. This is what keeps the API key out of the code. |

A `ModuleNotFoundError` here means the notebook is running against a Python that does not
have `requests` installed — almost always the wrong kernel. The next cell reports which
interpreter is actually active.

In [2]:
# Which Python is running this notebook? Should resolve inside .venv/.
print(sys.executable)

/Users/pratikpatel/Library/CloudStorage/OneDrive-RutgersUniversity/Projects/Data_Engineering_Project_1/.venv/bin/python


If that path does not contain `.venv`, the kernel is wrong. Registering the project's
virtual environment as a Jupyter kernel:

```bash
source .venv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name de-project-1 --display-name "Python (DE Project 1)"
```

Then **Kernel -> Change Kernel -> Python (DE Project 1)**.

`ipykernel` is what registers a virtual environment as a kernel Jupyter is allowed to run
notebooks against. Without it, Jupyter falls back to a system Python that has none of the
project's packages installed.

---
## 2. Keeping the key out of the file

An API key is a password. Anyone holding it can spend the rate limit, and on a paid tier,
real money. One rule follows from that:

> **A secret never appears in a file that gets committed to git.**

The mechanism is an **environment variable** — a named value belonging to the running
process rather than to any file of source code. The key lives in `.env`:

```
POLYGON_API_KEY=abc123...
```

`load_dotenv()` reads that file into the environment; `os.getenv("POLYGON_API_KEY")` asks
the environment for it. The code names the value and never contains it.

`.env` is listed in `.gitignore`. `.env.example` sits beside it holding a placeholder and
*is* committed on purpose — it tells anyone cloning this repo which variables to supply.

This is not ceremony. Leaked API keys in public repositories are common enough that
scanners crawl for them within minutes of a push.

> The variable is named `POLYGON_API_KEY` because Polygon.io was the provider's name at the
> time. It rebranded to **Massive** in October 2025, but `api.polygon.io` and existing keys
> still work. Renaming the variable would mean editing two files for no functional gain.

In [3]:
# load_dotenv() looks for a file named .env in this folder (and parent folders)
# and loads it. It returns True if it found one.
found = load_dotenv()
print("Found a .env file:", found)

API_KEY = os.getenv("POLYGON_API_KEY")

# Print whether the key loaded, never the key itself.
# Never print(API_KEY) while debugging -- terminal output and notebook
# output both get screenshotted, and a screenshotted key is a rotated key.
print("Key loaded:", bool(API_KEY))
print("Key length:", len(API_KEY) if API_KEY else 0, "characters")

from safety import guard_secrets
guard_secrets()

Found a .env file: True
Key loaded: True
Key length: 32 characters
Screenshot guard ON. 1 secret(s) will render as <API-KEY-HIDDEN>.
Print anything you like. Screenshots are safe.


Expect `True`, `True`, and a length around 30–35 characters.

`Found a .env file: False` means the notebook's working directory is not the project root.
The next cell reports where it actually is.

In [4]:
print("Working directory:", os.getcwd())
print("Files here:", sorted(f for f in os.listdir() if not f.startswith(".DS")))

Working directory: /Users/pratikpatel/Library/CloudStorage/OneDrive-RutgersUniversity/Projects/Data_Engineering_Project_1/notebooks
Files here: ['__pycache__', 'safety.py', 'step_2_first_api_call.ipynb', 'step_3_ticker_loop.ipynb']


---
## 3. What the request actually is

Stripped of vocabulary, an API call is one sentence: a program asks a server for something
over the internet, and the server answers.

The protocol is **HTTP** — the same one a browser uses. A browser sends an HTTP request and
gets HTML back; this code sends the same kind of request and gets **JSON** back instead,
because JSON is meant for programs and HTML is meant for eyes.

**JSON** (JavaScript Object Notation) maps directly onto Python dictionaries and lists.
`{"ticker": "AAPL", "close": 310.34}` becomes a dict with two keys, and the conversion is a
single method call.

### Anatomy of the URL

```
https://api.polygon.io/v2/aggs/ticker/AAPL/prev?adjusted=true&apiKey=abc123
└──┬──┘└──────┬───────┘└──────────┬──────────┘ └──────────────┬────────────┘
scheme     host              path                     query parameters
```

- **scheme** — `https`, meaning encrypted. Mandatory for anything carrying a key.
- **host** — which server to talk to: `api.polygon.io`.
- **path** — *which* resource. Here: aggregate bars, ticker AAPL, previous day. It reads
  left to right as a hierarchy, like folders.
- **query parameters** — the options, after the `?`, joined by `&`.

An **endpoint** is a specific path that does a specific job. `/v2/aggs/ticker/{t}/prev`
returns the previous trading day's summary. A different path returns a date range — that is
the one step 3 uses.

### GET

HTTP requests have **methods**: verbs describing intent. `GET` means "return something,
change nothing." `POST` means "here is data, do something with it." Reading market data is a
`GET`, which is why the call is `requests.get()`.

### Bars

Market data arrives in **bars** (also called candles): one row summarising one time period.
A daily bar is one day of trading condensed into open, high, low, close and volume —
conventionally abbreviated **OHLCV**. The previous-day endpoint returns exactly one bar;
step 3's range endpoint returns many.

---
## 4. Configuration

Constants at the top, capitalised by convention — a signal to a reader that these are fixed
values rather than variables that change as the program runs.

In [5]:
BASE_URL = "https://api.polygon.io"
TICKER = "AAPL"

print(BASE_URL, TICKER)

https://api.polygon.io AAPL


---
## 5. Building the URL

The previous-day path is `/v2/aggs/ticker/{TICKER}/prev`, assembled with an f-string.

**The API key is deliberately absent from this URL.** It goes into the parameters in the
next cell, which means the URL itself can be printed while debugging without exposing
anything. Section 12.5 revisits that decision and moves the key further still — out of the
query string entirely.

In [6]:
# Build the previous-day endpoint URL. The key is deliberately not in it.
url = f"{BASE_URL}/v2/aggs/ticker/{TICKER}/prev"

print(url)
# -> https://api.polygon.io/v2/aggs/ticker/AAPL/prev

https://api.polygon.io/v2/aggs/ticker/AAPL/prev


---
## 6. Query parameters

`requests` takes a dictionary and builds the `?key=value&key=value` portion of the URL
itself, escaping awkward characters along the way. Building that string by hand is never
necessary.

| Key | Value | Purpose |
|---|---|---|
| `apiKey` | the key | Authentication. Note the capital **K** — a lowercase `k` returns 401. |
| `adjusted` | `"true"` | Corrects prices for stock splits. |

**On `adjusted`, which is easy to skip past and shouldn't be.** When a company does a
2-for-1 split, every share becomes two shares at half the price. Nothing about the company's
value changed, but the raw price series shows an overnight 50% crash. Adjusted prices
rewrite the history so the split does not read as a collapse. Apple has split five times.
A chart built on unadjusted data is wrong in a way that still looks plausible, which is the
worst kind of wrong.

The value is the **string** `"true"`, not Python's `True` — it is going into a URL, and URLs
carry text.

In [7]:
# Query parameters. requests turns this dict into the ?a=b&c=d portion.
params = {"apiKey": API_KEY, "adjusted": "true"}

# Print the keys only -- printing the dict would print the key itself.
print("Parameter names:", list(params.keys()) if isinstance(params, dict) else "not a dict yet")

Parameter names: ['apiKey', 'adjusted']


---
## 7. Making the request

`requests.get()` takes the URL, the parameters, and a timeout.

**The timeout is not optional.** Without it, a server that accepts the connection and then
never answers leaves the program waiting indefinitely — no error, no output, just a cell
that never finishes. In a scheduled pipeline that is a job hanging silently until somebody
notices days later. Ten seconds is generous for a small JSON response.

This is the cell that spends one of the five calls per minute.

In [8]:
# One GET. This spends one of the five calls per minute.
response = requests.get(url, params=params, timeout=10)

print("Type:", type(response))
print("HTTP status code:", response.status_code)

Type: <class 'requests.models.Response'>
HTTP status code: 200


---
## 8. Reading the response

What comes back is a `Response` object: the whole reply, envelope included, not yet the
data. Three parts are worth knowing.

**The status code** — three digits describing how it went:

| Code | Meaning | Response |
|---|---|---|
| **200** | OK | Continue. |
| **401** | Unauthorized | Key rejected. Check `.env`. |
| **403** | Forbidden | Key is valid but not entitled to this data — usually a paid-tier endpoint. |
| **404** | Not Found | Bad path, or a ticker that does not exist. |
| **429** | Too Many Requests | Rate limited. Wait a minute. |
| **5xx** | Server error | Their side, not the caller's. Retry later. |

The families are the part worth remembering: **2xx** succeeded, **4xx** the caller got
something wrong, **5xx** the server did.

**The headers** — metadata about the reply, as a dictionary. Some APIs report the remaining
rate-limit budget here, which step 3 makes use of.

**The body** — the actual data, as raw text until it is parsed.

In [9]:
print("Status:", response.status_code, response.reason)
print()
print("--- Headers ---")
for k, v in response.headers.items():
    print(f"{k}: {v}")

Status: 200 OK

--- Headers ---
Date: Thu, 27 Aug 2026 23:10:45 GMT
Content-Type: application/json
Transfer-Encoding: chunked
Connection: keep-alive
Vary: Accept-Encoding, Accept-Encoding
X-Polygon-Cluster-Name: polygon-ny5
X-Request-Id: 44accff67d7dde3a3cc77f3f514e551a
Strict-Transport-Security: max-age=15724800; includeSubDomains
Content-Encoding: gzip


In [10]:
# The raw body, as text. This is what actually came down the wire --
# one long line of JSON. Readable, barely. This is why we parse it.
print(response.text[:600])

{"ticker":"AAPL","queryCount":1,"resultsCount":1,"adjusted":true,"results":[{"T":"AAPL","v":3.4024486e+07,"vw":313.235,"o":310.3,"c":313.45,"h":315.43,"l":308.8001,"t":1787774400000,"n":660791}],"status":"OK","request_id":"44accff67d7dde3a3cc77f3f514e551a","count":1}


---
## 9. Parsing the JSON

`.json()` on the response object turns the body text into a Python dictionary in one call,
with no arguments.

Once parsed, `data["results"]` is a real list and `data["ticker"]` is a real string — no
string slicing, no regular expressions.

In [11]:
# Parse the JSON body into a Python dict.
data = response.json()

print("Type:", type(data))
print("Top-level keys:", list(data.keys()))

Type: <class 'dict'>
Top-level keys: ['ticker', 'queryCount', 'resultsCount', 'adjusted', 'results', 'status', 'request_id', 'count']


---
## 10. Inspecting the response shape

Looking at the shape of a response before writing code against it is worth the two minutes.
It replaces twenty minutes of guessing at key names.

In [12]:
import json

# json.dumps with indent=2 pretty-prints a dict. Purely for reading.
print(json.dumps(data, indent=2))

{
  "ticker": "AAPL",
  "queryCount": 1,
  "resultsCount": 1,
  "adjusted": true,
  "results": [
    {
      "T": "AAPL",
      "v": 34024486.0,
      "vw": 313.235,
      "o": 310.3,
      "c": 313.45,
      "h": 315.43,
      "l": 308.8001,
      "t": 1787774400000,
      "n": 660791
    }
  ],
  "status": "OK",
  "request_id": "44accff67d7dde3a3cc77f3f514e551a",
  "count": 1
}


A response of roughly this shape comes back:

```json
{
  "ticker": "AAPL",
  "queryCount": 1,
  "resultsCount": 1,
  "adjusted": true,
  "results": [
    {
      "T": "AAPL",
      "v": 44988230,
      "vw": 309.8817,
      "o": 308.5,
      "c": 310.34,
      "h": 311.2,
      "l": 307.8,
      "t": 1756080000000,
      "n": 512340
    }
  ],
  "status": "OK",
  "request_id": "a1b2c3..."
}
```

**The single-letter keys inside `results`:**

| Key | Meaning |
|---|---|
| `T` | Ticker |
| `o` | **O**pen — first trade of the day |
| `h` | **H**igh |
| `l` | **L**ow |
| `c` | **C**lose — last trade of the day. This is the value this notebook is after. |
| `v` | **V**olume — shares traded |
| `vw` | **V**olume-**w**eighted average price |
| `t` | **T**imestamp, in epoch milliseconds |
| `n` | **N**umber of transactions |

They are abbreviated because a full year of daily bars for 30 tickers is roughly 7,500 of
these objects, and `"c"` versus `"close"` saves real bandwidth at that scale. Renaming them
into something human-readable is precisely what the dbt staging model in step 7 does.

**`status` and `request_id` are worth noticing.** `status` is the API's *own* verdict,
separate from the HTTP code — a `200 OK` can arrive carrying a body with no data in it.
Checking both is the habit. `request_id` is the value to quote when contacting support.

In [13]:
# Pull out the one bar and look at it on its own.
results = data["results"]
print("Number of bars returned:", len(results))

bar = results[0]   # a list, even with one item -- the same endpoint shape scales to many
print()
for k, v in bar.items():
    print(f"  {k!r:6} -> {v}")

Number of bars returned: 1

  'T'    -> AAPL
  'v'    -> 34024486.0
  'vw'   -> 313.235
  'o'    -> 310.3
  'c'    -> 313.45
  'h'    -> 315.43
  'l'    -> 308.8001
  't'    -> 1787774400000
  'n'    -> 660791


---
## 11. The timestamp

`"t": 1756080000000` is a **Unix timestamp in milliseconds** — milliseconds elapsed since
midnight UTC on 1 January 1970. That moment is the *epoch*, and effectively every computer
counts time from it. Being a single integer, it sorts and compares trivially and carries no
timezone ambiguity.

Python's `datetime.fromtimestamp()` expects **seconds** and the API sends **milliseconds**.
Dividing by 1000 is the entire fix, and forgetting it is the classic first encounter with
epoch time: 1,756,080,000,000 seconds after 1970 lands somewhere around the year 57,600.

`tz=timezone.utc` says *interpret this in UTC*. Left out, Python quietly uses the machine's
local timezone, which shifts some dates by a day — in a pipeline, that mislabels which
trading day a price belongs to, and the error survives all the way to a chart.

In [14]:
def ms_epoch_to_date(ms: int) -> str:
    # Convert epoch milliseconds to a 'YYYY-MM-DD' string in UTC.
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).strftime("%Y-%m-%d")


# Demonstration of why the / 1000 matters:
raw = bar["t"]
print("Raw value:      ", raw)
print("Correct  (/1000):", ms_epoch_to_date(raw))
print("Wrong (no /1000):", datetime.fromtimestamp(raw / 1000 / 1000, tz=timezone.utc).strftime("%Y-%m-%d"), "<- treating ms as seconds would be even further off")

Raw value:       1787774400000
Correct  (/1000): 2026-08-26
Wrong (no /1000): 1970-01-21 <- treating ms as seconds would be even further off


---
## 12. Extracting the close price

`bar["c"]` holds the close and `bar["t"]` the timestamp. The timestamp goes through
`ms_epoch_to_date()`. The format spec `{value:.2f}` rounds a float to two decimal places,
which is what money requires — `310.34`, not `310.3400000000001`.

Output:

```
AAPL closed at $310.34 on 2026-08-24
```

In [15]:
# Pull the close and the trade date out of the bar, and format them.
close_price = bar["c"]
trade_date = ms_epoch_to_date(bar["t"])

print(f"{TICKER} closed at ${close_price:.2f} on {trade_date}")

AAPL closed at $313.45 on 2026-08-26


### Step 2 is functionally complete at this point.

Data has moved from a market data provider's servers into this process. Everything after
this is volume, storage and shape — the question of whether the data is reachable at all is
now answered.

---
## 12.5 — The key turned up in an exception message

I ran this code in an environment with no outbound network access. `requests` raised a
connection error, and the message read:

```
ProxyError: HTTPSConnectionPool(host='api.polygon.io', port=443):
Max retries exceeded with url: /v2/aggs/ticker/AAPL/prev?apiKey=<the key, in full>&adjusted=true
```

**The key is in the exception text.** Not in anything I printed deliberately — in the
automatic wording of an error nobody chose.

This is worth more than the ten minutes it costs to fix, because it is neither obvious nor
rare:

- `requests` assembles the final URL from `url` + `params`, then quotes that **full URL**,
  query string included, in most of its exception messages.
- Tracebacks go to logs. By step 4 those are files on disk; by step 10 an orchestrator
  captures them and serves them in a web UI. A secret in a traceback reaches every one of
  those places, and nobody thinks to go looking for it there.
- Nothing here is a mistake in the ordinary sense. The `?apiKey=` pattern is exactly what
  the API's own documentation shows. It leaks by design.

### The fix: send the key in a header

HTTP requests carry **headers** — metadata sent alongside the request, separate from the
URL. Authentication belongs there, and `Authorization: Bearer <key>` is the standard form.
This API accepts it.

```python
headers = {"Authorization": f"Bearer {API_KEY}"}
response = requests.get(url, params={"adjusted": "true"}, headers=headers, timeout=10)
```

The URL is now `https://api.polygon.io/v2/aggs/ticker/AAPL/prev?adjusted=true` and carries
no secret. Headers are not echoed in exception messages, so tracebacks, logs and anything
else that quotes the URL are all safe.

**This is the version I kept.** Sections 5 to 7 above use the query-parameter form
deliberately, because it is what the documentation shows and what appears in essentially
every tutorial — it is worth being able to recognise. But the function carried into
`fetch_one_ticker.py`, and into every step after it, uses the header, and section 13 below
is written that way.

In [16]:
 # The same call with the key in a header. Compare the printed URL to section 5's.
headers = {"Authorization": f"Bearer {API_KEY}"}
safe_response = requests.get(
    f"{BASE_URL}/v2/aggs/ticker/{TICKER}/prev",
    params={"adjusted": "true"},
    headers=headers,
    timeout=10,
)

print("HTTP:", safe_response.status_code)
# .url is the exact URL that went out. Safe to print now -- no key in it.
print("URL actually requested:", safe_response.url)

HTTP: 200
URL actually requested: https://api.polygon.io/v2/aggs/ticker/AAPL/prev?adjusted=true


---
## 13. Handling the failure paths

The code above works when everything goes right. Production code is mostly the other
branches. Below is the same logic with the failure paths handled, and this is the version
that graduates into `fetch_one_ticker.py`.

Three ideas in it recur throughout step 3:

1. **Fail loudly and early.** If the key is missing, say *the key is missing* — do not let
   the program continue and surface a confusing 401 four lines later. Vague errors are how
   an evening disappears.
2. **Check the body, not just the status code.** A `200` carrying `"status": "ERROR"`, or an
   empty `results` list, is a failure wearing a success costume. Markets close on weekends
   and holidays, so the empty case is normal rather than exceptional.
3. **Name the errors that are expected.** A 429 is a certainty on a 5-calls-per-minute tier.
   A generic "request failed" means diagnosing it from scratch every time; a message that
   says *rate limited, wait a minute* turns it into a non-event.

In [17]:
def fetch_previous_close(ticker: str) -> dict:
    # Fetch the previous trading day's bar for one ticker.
    # Returns a dict with keys: ticker, date, close.
    # Raises RuntimeError with a readable message on any failure.
    api_key = os.getenv("POLYGON_API_KEY")
    if not api_key:
        raise RuntimeError(
            "POLYGON_API_KEY not found. Is there a .env file in this folder "
            "with a line like POLYGON_API_KEY=your_key_here ?"
        )

    url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/prev"

    # Key goes in the header, NOT in params -- see section 12.5.
    # Anything in params can end up in an exception message or a log line.
    headers = {"Authorization": f"Bearer {api_key}"}
    params = {"adjusted": "true"}

    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
    except requests.exceptions.Timeout:
        raise RuntimeError(f"Request for {ticker} timed out after 10 seconds.")
    except requests.exceptions.ConnectionError:
        raise RuntimeError(f"Could not reach the API for {ticker}. Check your connection.")

    if response.status_code == 429:
        raise RuntimeError("Rate limited (429). Free tier is 5 calls/min -- wait 60 seconds.")
    if response.status_code == 401:
        raise RuntimeError("Unauthorized (401). The API key was rejected -- check .env.")
    if response.status_code == 403:
        raise RuntimeError("Forbidden (403). Key is valid but not entitled to this endpoint.")

    # Anything else in the 4xx/5xx range becomes an exception here.
    response.raise_for_status()

    payload = response.json()

    if payload.get("status") not in ("OK", "DELAYED"):
        raise RuntimeError(f"API reported status={payload.get('status')!r}: {payload}")

    bars = payload.get("results") or []
    if not bars:
        raise RuntimeError(
            f"No bars returned for {ticker!r}. "
            f"HTTP {response.status_code}, api status={payload.get('status')!r}, "
            f"queryCount={payload.get('queryCount')}, "
            f"resultsCount={payload.get('resultsCount')}. "
            "Check the symbol is real before assuming a market-calendar issue."
        )

    bar = bars[0]
    return {
        "ticker": ticker,
        "date": ms_epoch_to_date(bar["t"]),
        "close": bar["c"],
    }

In [18]:
# One call, cleanly. (Spends another of the 5-per-minute budget.)
result = fetch_previous_close("AAPL")
print(f"{result['ticker']} closed at ${result['close']:.2f} on {result['date']}")

AAPL closed at $313.45 on 2026-08-26


---
## 14. Confirming the error handling works

Requesting a ticker that does not exist should produce the readable message defined above
rather than a raw traceback. **Costs one API call.**

In [19]:
try:
    fetch_previous_close("NOTAREALTICKER")
except RuntimeError as e:
    print("Caught cleanly:", e)

Caught cleanly: No bars returned for 'NOTAREALTICKER'. HTTP 200, api status='OK', queryCount=0, resultsCount=0. Check the symbol is real before assuming a market-calendar issue.


In [20]:
for t in ("AAPL", "NOTAREALTICKER"):
    r = requests.get(
        f"{BASE_URL}/v2/aggs/ticker/{t}/prev",
        params={"adjusted": "true"}, headers=headers, timeout=10,
    )
    d = r.json()
    print(f"{t:<16} HTTP {r.status_code} | status={d.get('status')!r} "
          f"| queryCount={d.get('queryCount')} | resultsCount={d.get('resultsCount')} "
          f"| results={d.get('results')!r}")

AAPL             HTTP 200 | status='OK' | queryCount=1 | resultsCount=1 | results=[{'T': 'AAPL', 'v': 34024486.0, 'vw': 313.235, 'o': 310.3, 'c': 313.45, 'h': 315.43, 'l': 308.8001, 't': 1787774400000, 'n': 660791}]
NOTAREALTICKER   HTTP 429 | status='ERROR' | queryCount=None | resultsCount=None | results=None


In [21]:
r = requests.get(
    f"{BASE_URL}/v2/aggs/ticker/AAPL/range/1/day/2026-08-21/2026-08-26",
    params={"adjusted": "true", "sort": "asc"},
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=10,
)
d = r.json()
print("HTTP", r.status_code, "| resultsCount:", d.get("resultsCount"))
for b in d.get("results", []):
    print(ms_epoch_to_date(b["t"]), "close", b["c"])

HTTP 429 | resultsCount: None


---
## 15. What step 3 needs

Step 2 asks whether one price is reachable. Step 3 asks whether many prices, for many
tickers, can be fetched without being throttled or losing data halfway through.

That is a different endpoint — a date range rather than a single day:

```
/v2/aggs/ticker/{ticker}/range/1/day/{from_date}/{to_date}
```

The response has the same shape, but `results` carries one bar per trading day instead of
one bar in total.

The interesting part of step 3 is not the endpoint. It is what surrounds it: 25 tickers
against a 5-per-minute limit means the loop has to pace itself, survive a failure on ticker
17 without losing tickers 1–16, and write files in a layout that is still navigable six
weeks later.

The working code from section 13 — `fetch_previous_close` — moves into `fetch_one_ticker.py`
and runs from the terminal before step 3 begins. A notebook that works and a script that
works are two different claims.